# Training an Encrypted Neural Network

In this tutorial, we will walk through an example of how we can train a neural network with CrypTen. This is particularly relevant for the <i>Feature Aggregation</i>, <i>Data Labeling</i> and <i>Data Augmentation</i> use cases. We will focus on the usual two-party setting and show how we can train an accurate neural network for digit classification on the MNIST data.

For concreteness, this tutorial will step through the <i>Feature Aggregation</i> use cases: Alice and Bob each have part of the features of the data set, and wish to train a neural network on their combined data, while keeping their data private. 

## Setup
As usual, we'll begin by importing and initializing the `crypten` and `torch` libraries.  

We will use the MNIST dataset to demonstrate how Alice and Bob can learn without revealing protected information. For reference, the feature size of each example in the MNIST data is `28 x 28`. Let's assume Alice has the first `28 x 20` features and Bob has last `28 x 8` features. One way to think of this split is that Alice has the (roughly) top 2/3rds of each image, while Bob has the bottom 1/3rd of each image. We'll again use our helper script `mnist_utils.py` that downloads the publicly available MNIST data, and splits the data as required.

For simplicity, we will restrict our problem to binary classification: we'll simply learn how to distinguish between 0 and non-zero digits. For speed of execution in the notebook, we will only create a dataset of a 100 examples.

In [1]:
import crypten
import torch


crypten.init()
torch.set_num_threads(1)

[W ProcessGroupGloo.cpp:723] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())


In [2]:
%run ./mnist_utils.py --option features --reduced 100 --binary

/opt/homebrew/Caskroom/miniconda/base/envs/lora_mps/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 9912422/9912422 [00:02<00:00, 4178220.31it/s]


Extracting /tmp/MNIST/raw/train-images-idx3-ubyte.gz to /tmp/MNIST/raw



100%|██████████| 28881/28881 [00:00<00:00, 1232068.00it/s]


Extracting /tmp/MNIST/raw/train-labels-idx1-ubyte.gz to /tmp/MNIST/raw



100%|██████████| 1648877/1648877 [00:01<00:00, 1182899.85it/s]


Extracting /tmp/MNIST/raw/t10k-images-idx3-ubyte.gz to /tmp/MNIST/raw



100%|██████████| 4542/4542 [00:00<00:00, 1574944.51it/s]


Extracting /tmp/MNIST/raw/t10k-labels-idx1-ubyte.gz to /tmp/MNIST/raw



Next, we'll define the network architecture below, and then describe how to train it on encrypted data in the next section. 

In [3]:
import torch.nn as nn
import torch.nn.functional as F

#Define an example network
class ExampleNet(nn.Module):
    def __init__(self):
        super(ExampleNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=0)
        self.fc1 = nn.Linear(16 * 12 * 12, 100)
        self.fc2 = nn.Linear(100, 2) # For binary classification, final layer needs only 2 outputs
 
    def forward(self, x):
        out = self.conv1(x)
        out = F.relu(out)
        out = F.max_pool2d(out, 2)
        out = out.view(-1, 16 * 12 * 12)
        out = self.fc1(out)
        out = F.relu(out)
        out = self.fc2(out)
        return out
    
crypten.common.serial.register_safe_class(ExampleNet)

## Encrypted Training

After all the material we've covered in earlier tutorials, we only need to know a few additional items for encrypted training. We'll first discuss how the training loop in CrypTen differs from PyTorch. Then, we'll go through a complete example to illustrate training on encrypted data from end-to-end.

### How does CrypTen training differ from PyTorch training?

There are two main ways implementing a CrypTen training loop differs from a PyTorch training loop. We'll describe these items first, and then illustrate them with small examples below.

<i>(1) Use one-hot encoding</i>: CrypTen training requires all labels to use one-hot encoding. This means that when using standard datasets such as MNIST, we need to modify the labels to use one-hot encoding.

<i>(2) Directly update parameters</i>: CrypTen does not use the PyTorch optimizers. Instead, CrypTen implements encrypted SGD by implementing its own `backward` function, followed by directly updating the parameters. As we will see below, using SGD in CrypTen is very similar to using the PyTorch optimizers.

We now show some small examples to illustrate these differences. As before, we will assume Alice has the rank 0 process and Bob has the rank 1 process.

In [4]:
# Define source argument values for Alice and Bob
ALICE = 0
BOB = 1

In [5]:
# Load Alice's data 
data_alice_enc = crypten.load_from_party('/tmp/alice_train.pth', src=ALICE)

In [6]:
# We'll now set up the data for our small example below
# For illustration purposes, we will create toy data
# and encrypt all of it from source ALICE
x_small = torch.rand(100, 1, 28, 28)
print(x_small)
y_small = torch.randint(1, (100,))

# Transform labels into one-hot encoding
label_eye = torch.eye(2)
y_one_hot = label_eye[y_small]

# Transform all data to CrypTensors
x_train = crypten.cryptensor(x_small, src=ALICE)
y_train = crypten.cryptensor(y_one_hot)


# Instantiate and encrypt a CrypTen model
model_plaintext = ExampleNet()
dummy_input = torch.empty(1, 1, 28, 28)
model = crypten.nn.from_pytorch(model_plaintext, dummy_input)
model.encrypt()

tensor([[[[1.4721e-01, 6.0200e-01, 5.7154e-01,  ..., 7.4211e-02,
           6.1085e-01, 3.1913e-01],
          [6.8153e-02, 4.5589e-01, 6.5478e-01,  ..., 6.3890e-01,
           5.0505e-01, 7.3355e-01],
          [4.8886e-01, 7.2510e-01, 7.4453e-01,  ..., 1.4909e-01,
           2.0083e-01, 4.9624e-01],
          ...,
          [6.7610e-01, 3.9845e-01, 7.2964e-01,  ..., 4.7268e-02,
           5.6744e-01, 4.4802e-01],
          [9.7475e-01, 7.4722e-01, 4.2948e-02,  ..., 4.3908e-01,
           6.4399e-01, 6.3512e-01],
          [7.8211e-01, 6.1490e-01, 1.7966e-02,  ..., 2.3223e-01,
           4.4571e-01, 6.0645e-01]]],


        [[[3.1855e-01, 2.1521e-01, 2.3760e-01,  ..., 7.7893e-01,
           5.7658e-01, 9.0821e-01],
          [5.4579e-01, 3.3671e-01, 6.0328e-01,  ..., 7.6890e-01,
           2.7621e-01, 8.3217e-02],
          [6.2934e-01, 7.1149e-01, 4.0771e-01,  ..., 4.8065e-01,
           3.8691e-02, 9.1198e-01],
          ...,
          [4.1203e-01, 9.3058e-01, 6.7930e-01,  ..., 2.87

/opt/homebrew/Caskroom/miniconda/base/envs/lora_mps/lib/python3.8/site-packages/crypten-0.4.0-py3.8.egg/crypten/nn/onnx_converter.py:176: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:212.)
  param = torch.from_numpy(numpy_helper.to_array(node))


Graph encrypted module

In [7]:
# Example: Stochastic Gradient Descent in CrypTen

model.train() # Change to training mode
loss = crypten.nn.MSELoss() # Choose loss functions

# Set parameters: learning rate, num_epochs
learning_rate = 0.001
num_epochs = 2
optimizer = crypten.optim.DPSGD(model.parameters(), learning_rate)
# Train the model: SGD on encrypted data
for i in range(num_epochs):

    # forward pass
    output = model(x_train)
    loss_value = loss(output, y_train)
    
    
    # set gradients to zero
    model.zero_grad()

    # perform backward pass
    loss_value.backward()

    optimizer.step()
    # update parameters
    #model.update_parameters(learning_rate) 
    
    # examine the loss after each epoch
    print("Epoch: {0:d} Loss: {1:.4f}".format(i, loss_value.get_plain_text()))



TypeError: __init__() missing 2 required positional arguments: 'noise_local_stddev' and 'l2_clipping_threshold'

### A Complete Example

We now put these pieces together for a complete example of training a network in a multi-party setting. 

As in Tutorial 3, we'll assume Alice has the rank 0 process, and Bob has the rank 1 process; so we'll load and encrypt Alice's data with `src=0`, and load and encrypt Bob's data with `src=1`. We'll then initialize a plaintext model and convert it to an encrypted model, just as we did in Tutorial 4. We'll finally define our loss function, training parameters, and run SGD on the encrypted data. For the purposes of this tutorial we train on 100 samples; training should complete in ~3 minutes per epoch.

In [19]:
import crypten.mpc as mpc
import crypten.communicator as comm

# Convert labels to one-hot encoding
# Since labels are public in this use case, we will simply use them from loaded torch tensors
labels = torch.load('/tmp/train_labels.pth')
labels = labels.long()
labels_one_hot = label_eye[labels]

@mpc.run_multiprocess(world_size=2)
def run_encrypted_training():
    # Load data:
    x_alice_enc = crypten.load_from_party('/tmp/alice_train.pth', src=ALICE)
    x_bob_enc = crypten.load_from_party('/tmp/bob_train.pth', src=BOB)

    
    # Combine the feature sets: identical to Tutorial 3
    x_combined_enc = crypten.cat([x_alice_enc, x_bob_enc], dim=2)
    
    # Reshape to match the network architecture
    x_combined_enc = x_combined_enc.unsqueeze(1)
    
    
    # Commenting out due to intermittent failure in PyTorch codebase
    
    # Initialize a plaintext model and convert to CrypTen model
    pytorch_model = ExampleNet()
    model = crypten.nn.from_pytorch(pytorch_model, dummy_input)
    model.encrypt()
    # Set train mode
    model.train()
  
    # Define a loss function
    loss = crypten.nn.MSELoss()

    # Define training parameters
    learning_rate = 0.001
    num_epochs = 2
    batch_size = 1
    num_batches = x_combined_enc.size(0) // batch_size
    optimizer = crypten.optim.DPSGD(model.parameters(), learning_rate)
    
    rank = comm.get().get_rank()
    for i in range(num_epochs): 
        crypten.print(f"Epoch {i} in progress:")       
        
        for batch in range(num_batches):
            # define the start and end of the training mini-batch
            start, end = batch * batch_size, (batch + 1) * batch_size
                                    
            # construct CrypTensors out of training examples / labels
            x_train = x_combined_enc[start:end]
            y_batch = labels_one_hot[start:end]
            y_train = crypten.cryptensor(y_batch, requires_grad=True)

            # perform forward pass:
            output = model(x_train)
            loss_value = loss(output, y_train)
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value forward: {loss_value}", in_order=True)
            # set gradients to "zero" 
            model.zero_grad()

            # perform backward pass: 
            loss_value.backward()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tLoss value backward: {model.}", in_order=True)
            

            # update parameters
            #model.update_parameters(learning_rate)
            #loss_value.backward()

            optimizer.step()
            #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tBath: {batch}\n\tModel: {model}", in_order=True)

            
            # Print progress every batch:
            batch_loss = loss_value.get_plain_text()
            crypten.print(f"\tBatch {(batch + 1)} of {num_batches} Loss {batch_loss.item():.4f}")
        #crypten.print(f"Rank: {rank}\n\tEpoch: {i}\n\tLoss value: {batch_loss}", in_order=True)


run_encrypted_training()

Epoch 0 in progress:
	Batch 1 of 100 Loss 0.8504
	Batch 2 of 100 Loss 0.3889
	Batch 3 of 100 Loss 0.5120
	Batch 4 of 100 Loss 0.5106
	Batch 5 of 100 Loss 0.5489
	Batch 6 of 100 Loss 0.4020
	Batch 7 of 100 Loss 0.6223
	Batch 8 of 100 Loss 0.4806
	Batch 9 of 100 Loss 0.5580
	Batch 10 of 100 Loss 0.4969
	Batch 11 of 100 Loss 0.5666
	Batch 12 of 100 Loss 0.4020
	Batch 13 of 100 Loss 0.6414
	Batch 14 of 100 Loss 0.3335
	Batch 15 of 100 Loss 0.4325
	Batch 16 of 100 Loss 0.3305
	Batch 17 of 100 Loss 0.3024
	Batch 18 of 100 Loss 0.1925
	Batch 19 of 100 Loss 0.3830
	Batch 20 of 100 Loss 0.3706
	Batch 21 of 100 Loss 0.3001
	Batch 22 of 100 Loss 0.4276
	Batch 23 of 100 Loss 0.1929
	Batch 24 of 100 Loss 0.1882
	Batch 25 of 100 Loss 0.3370
	Batch 26 of 100 Loss 0.1011
	Batch 27 of 100 Loss 0.2145
	Batch 28 of 100 Loss 0.1341
	Batch 29 of 100 Loss 0.1129
	Batch 30 of 100 Loss 0.2070
	Batch 31 of 100 Loss 0.3060
	Batch 32 of 100 Loss 0.0489
	Batch 33 of 100 Loss 0.2467
	Batch 34 of 100 Loss 0.0881
	B

[None, None]

We see that the average batch loss decreases across the epochs, as we expect during training.

This completes our tutorial. Before exiting this tutorial, please clean up the files generated using the following code.

In [20]:
import os

filenames = ['/tmp/alice_train.pth', 
             '/tmp/bob_train.pth', 
             '/tmp/alice_test.pth',
             '/tmp/bob_test.pth', 
             '/tmp/train_labels.pth',
             '/tmp/test_labels.pth']

for fn in filenames:
    if os.path.exists(fn): os.remove(fn)